Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from warnings import filterwarnings
filterwarnings("ignore")

Load Dataset

In [ ]:
df=pd.read_csv("/content/CreditRisk.csv")
df.head()

In [ ]:
df.shape

Check Missing Values

In [ ]:
df.isnull().sum()[df.isnull().sum()>0]

Drop Unnecessary Column

In [ ]:
df=df.drop("Loan_ID",axis=1)

Encode Target Variable

In [ ]:
df['Loan_Status']=df['Loan_Status'].map({'Y':1,'N':0})

Train Test Split (done BEFORE handling missing values, so test data does not leak into training)

In [ ]:
from sklearn.model_selection import train_test_split

df_train,df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Loan_Status'])

df_train=df_train.copy()
df_test=df_test.copy()

Handle Missing Values (fill values are taken from the TRAIN data only, then applied to both train and test)

In [ ]:
mode_cols = ['Gender','Married','Dependents','Self_Employed','Loan_Amount_Term','Credit_History']

for col in mode_cols:
    fill_value = df_train[col].mode()[0]
    df_train[col] = df_train[col].fillna(fill_value)
    df_test[col] = df_test[col].fillna(fill_value)

loan_median = df_train['LoanAmount'].median()
df_train['LoanAmount'] = df_train['LoanAmount'].fillna(loan_median)
df_test['LoanAmount'] = df_test['LoanAmount'].fillna(loan_median)

Encode Categorical Variables

In [ ]:
df_train=pd.get_dummies(df_train,drop_first=True,dtype=int)
df_test=pd.get_dummies(df_test,drop_first=True,dtype=int)

# make sure test has exactly the same columns as train
df_test=df_test.reindex(columns=df_train.columns,fill_value=0)

Defining Features and Target

In [ ]:
df_train_x=df_train.drop("Loan_Status",axis=1)
df_train_y=df_train["Loan_Status"]

df_test_x=df_test.drop("Loan_Status",axis=1)
df_test_y=df_test["Loan_Status"]

Check Class Balance

In [ ]:
df_train_y.value_counts(normalize=True)

Training Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)

model.fit(df_train_x, df_train_y)

Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

pred = model.predict(df_test_x)

tab=confusion_matrix(df_test_y,pred)
print(tab)

acc=accuracy_score(df_test_y,pred)
print("Accuracy Score:",acc)

print(classification_report(df_test_y,pred))

Cross Validation

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(RandomForestClassifier(random_state=42), df_train_x, df_train_y, cv=5, scoring="f1")
print(cv_scores)
print("Mean F1:",cv_scores.mean())

Check Feature Names (the app must send inputs in exactly this order)

In [ ]:
print(list(df_train_x.columns))

Save Model

In [ ]:
import pickle

pickle.dump(model, open("loan_model.pkl","wb"))

Save requirements.txt (pins the scikit-learn version so the model loads on Streamlit Cloud)

In [ ]:
import sklearn

with open("requirements.txt","w") as f:
    f.write("streamlit\npandas\nnumpy\nscikit-learn=="+sklearn.__version__+"\n")

print(open("requirements.txt").read())

Creating Deployment App (Dependents input added)

In [ ]:
%%writefile app.py
import streamlit as st
import pickle
import pandas as pd

model = pickle.load(open("loan_model.pkl","rb"))

st.title("Loan Approval Prediction")

ApplicantIncome = st.number_input("Applicant Income")
CoapplicantIncome = st.number_input("Coapplicant Income")
LoanAmount = st.number_input("Loan Amount")
Loan_Amount_Term = st.number_input("Loan Amount Term", value=360.0)
Credit_History = st.selectbox("Credit History", [1,0])

Gender = st.selectbox("Gender", ["Male","Female"])
Married = st.selectbox("Married", ["Yes","No"])
Dependents = st.selectbox("Dependents", ["0","1","2","3+"])
Education = st.selectbox("Education", ["Graduate","Not Graduate"])
Self_Employed = st.selectbox("Self Employed", ["Yes","No"])
Property_Area = st.selectbox("Property Area", ["Urban","Semiurban","Rural"])

if st.button("Predict"):

    input_dict = {
        "ApplicantIncome": ApplicantIncome,
        "CoapplicantIncome": CoapplicantIncome,
        "LoanAmount": LoanAmount,
        "Loan_Amount_Term": Loan_Amount_Term,
        "Credit_History": Credit_History,
        "Gender_Male": 1 if Gender == "Male" else 0,
        "Married_Yes": 1 if Married == "Yes" else 0,
        "Dependents_1": 1 if Dependents == "1" else 0,
        "Dependents_2": 1 if Dependents == "2" else 0,
        "Dependents_3+": 1 if Dependents == "3+" else 0,
        "Education_Not Graduate": 1 if Education == "Not Graduate" else 0,
        "Self_Employed_Yes": 1 if Self_Employed == "Yes" else 0,
        "Property_Area_Semiurban": 1 if Property_Area == "Semiurban" else 0,
        "Property_Area_Urban": 1 if Property_Area == "Urban" else 0,
    }

    # arrange columns in the same order the model was trained on
    input_data = pd.DataFrame([input_dict])[model.feature_names_in_]

    prediction = model.predict(input_data)

    if prediction[0] == 1:
        st.success("Loan Approved")
    else:
        st.error("Loan Rejected")


Run in Colab with ngrok (optional). Put your token in Colab Secrets as NGROK_TOKEN. Never paste the token in the notebook or push it to GitHub.

In [ ]:
# !pip install streamlit pyngrok
# from google.colab import userdata
# from pyngrok import ngrok
#
# ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
# !streamlit run app.py &>/dev/null &
# public_url = ngrok.connect(8501)
# print(public_url)

In [ ]:
# from google.colab import files
# files.download("app.py")
# files.download("loan_model.pkl")
# files.download("requirements.txt")